In [21]:
import glob as glob
import pandas as pd
import os
import moviepy.editor as mpy
import random
import numpy as np
import uuid

def get_non_stroke_intervals(stroke_df, video_duration, min_gap=0.01):
    """
    Find intervals where no stroke occurs
    
    Args:
        stroke_df: DataFrame with stroke segments
        video_duration: Total duration of video in seconds
        min_gap: Minimum gap size to consider (seconds)
    
    Returns:
        List of (start, end) tuples for valid non-stroke intervals
    """
    # Convert milliseconds to seconds and sort segments
    segments = stroke_df[['start_segment', 'end_segment']].values / 1000
    segments = sorted(segments, key=lambda x: x[0])
    
    # Find gaps between strokes
    gaps = []
    last_end = 0
    
    for start, end in segments:
        if start - last_end >= min_gap:
            gaps.append((last_end, start))
        last_end = end
    
    # Add final gap if exists
    if video_duration - last_end >= min_gap:
        gaps.append((last_end, video_duration))
        
    return gaps

# folder
folder = './processed/'
outputfolder = './trainingsegments/'

# Create output folder if it doesn't exist
os.makedirs(outputfolder, exist_ok=True)

# annotation files and videos
files = glob.glob(folder + '*.csv')
videos = glob.glob(folder + '*.mp4')

# Column names
columns = ['tier', 'empty', 'start_segment', 'end_segment', 'label']

# Process each file
for csv_file in files:
    # Read and process CSV
    df = pd.read_csv(csv_file)
    df.columns = columns
    stroke_df = df[df['label'] == 'stroke'].copy()
    
    # Get corresponding video file
    video_file = csv_file.replace('csv', 'mp4')
    if not os.path.exists(video_file):
        print(f"Warning: Video file not found for {csv_file}")
        continue
    
    # Extract original video filename without extension
    original_video_name = os.path.splitext(os.path.basename(video_file))[0]
    
    # Get video duration
    video = mpy.VideoFileClip(video_file)
    video_duration = video.duration
    
    # Process stroke segments
    for idx, row in stroke_df.iterrows():
        duration = (row['end_segment'] - row['start_segment']) / 1000
        start = row['start_segment'] / 1000
        end = row['end_segment'] / 1000
        
        # Generate unique ID for this pair
        unique_id = str(uuid.uuid4())[:8]  # Using first 8 characters of UUID
        
        # Save stroke segment with original video name
        gesture_output = os.path.join(outputfolder, f'{original_video_name}_{unique_id}_Gesture.mp4')
        
        clip = video.subclip(start, end)
        clip.write_videofile(gesture_output)
        print(f'Saved gesture segment: {gesture_output}')
        
        # Find and save matching non-stroke segment
        non_stroke_intervals = get_non_stroke_intervals(stroke_df, video_duration)
        
        if non_stroke_intervals:
            # Choose random interval that's long enough
            valid_intervals = [interval for interval in non_stroke_intervals 
                             if interval[1] - interval[0] >= duration]
            
            if valid_intervals:
                chosen_interval = random.choice(valid_intervals)
                # Random start time within the chosen interval
                max_start = chosen_interval[1] - duration
                random_start = random.uniform(chosen_interval[0], max_start)
                random_end = random_start + duration
                
                # Save non-stroke segment with matching ID and original video name
                no_gesture_output = os.path.join(outputfolder, f'{original_video_name}_{unique_id}_NoGesture.mp4')
                
                clip = video.subclip(random_start, random_end)
                clip.write_videofile(no_gesture_output)
                print(f'Saved no-gesture segment: {no_gesture_output}')
            else:
                print(f'Warning: No valid non-stroke intervals of duration {duration:.2f}s found')
    
    # Close video to free resources
    video.close()

t:  50%|█████     | 2/4 [04:57<04:57, 148.55s/it, now=None]

Moviepy - Building video ./trainingsegments/M3D_TED_AS2010_e07c3698_Gesture.mp4.
MoviePy - Writing audio in M3D_TED_AS2010_e07c3698_GestureTEMP_MPY_wvf_snd.mp3


t:  50%|█████     | 2/4 [04:57<04:57, 148.58s/it, now=None]

MoviePy - Done.
Moviepy - Writing video ./trainingsegments/M3D_TED_AS2010_e07c3698_Gesture.mp4



t:  50%|█████     | 2/4 [04:57<04:57, 148.64s/it, now=None]

...

t:  50%|█████     | 2/4 [11:21<11:21, 340.54s/it, now=None]

Moviepy - Done !
Moviepy - video ready ./trainingsegments/M3D_TED_DL2016_07133e17_NoGesture.mp4
Saved no-gesture segment: ./trainingsegments/M3D_TED_DL2016_07133e17_NoGesture.mp4
Moviepy - Building video ./trainingsegments/M3D_TED_DL2016_42601ff4_Gesture.mp4.
MoviePy - Writing audio in M3D_TED_DL2016_42601ff4_GestureTEMP_MPY_wvf_snd.mp3


t:  50%|█████     | 2/4 [11:21<11:21, 340.58s/it, now=None]

MoviePy - Done.
Moviepy - Writing video ./trainingsegments/M3D_TED_DL2016_42601ff4_Gesture.mp4



OSError: [Errno 32] Broken pipe

MoviePy error: FFMPEG encountered the following error while writing file ./trainingsegments/M3D_TED_DL2016_42601ff4_Gesture.mp4:

 b'M3D_TED_DL2016_42601ff4_GestureTEMP_MPY_wvf_snd.mp3: Invalid data found when processing input\r\n'